# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.8: Práctica con GROMACS y OpenMM

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/08_practica_gromacs_openmm.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad (proyecto integrador), serás capaz de:
- Ejecutar de forma autónoma un protocolo completo de DM con GROMACS y/o OpenMM
- Comparar el desempeño y las capacidades de ambos softwares
- Analizar e interpretar los resultados de la simulación con herramientas integradas
- Evaluar la calidad de la simulación y detectar posibles artefactos
- Generar un informe científico con los resultados del análisis

---

## 1. Instalación de Dependencias

In [ ]:
!pip install numpy matplotlib MDAnalysis biopython requests seaborn pandas scipy
# GROMACS: conda install -c conda-forge gromacs
# OpenMM:  conda install -c conda-forge openmm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import subprocess
import os
from pathlib import Path

# Verificar disponibilidad de herramientas
def verificar_herramienta(nombre, comando):
    try:
        resultado = subprocess.run(
            comando, capture_output=True, text=True, timeout=5
        )
        return True
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

gromacs_ok = verificar_herramienta('GROMACS', ['gmx', '--version'])
print(f"GROMACS disponible: {'✓' if gromacs_ok else '✗ (instalar con: conda install -c conda-forge gromacs)'}")

try:
    import openmm
    print(f"OpenMM disponible: ✓ (versión {openmm.__version__})")
except ImportError:
    print("OpenMM disponible: ✗ (instalar con: conda install -c conda-forge openmm)")

## 2. Comparación GROMACS vs OpenMM

| Característica | GROMACS | OpenMM |
|---------------|---------|--------|
| **Lenguaje** | C++ (interfaz CLI) | Python API |
| **GPU** | CUDA, OpenCL, SYCL | CUDA, OpenCL, CPU |
| **Velocidad** | Muy alta | Alta |
| **Flexibilidad** | Media | Muy alta |
| **Campos de fuerza** | AMBER, CHARMM, GROMOS, OPLS | AMBER, CHARMM |
| **Análisis integrado** | Completo (gmx tools) | Limitado (usar MDAnalysis) |
| **Sistemas especiales** | Membranas, NMR, metadinámica | Custom forces fáciles |
| **Documentación** | Excelente | Excelente |
| **Licencia** | LGPL (libre) | MIT (libre) |
| **Curva de aprendizaje** | Alta | Media |

**Recomendación:** GROMACS para producción en HPC; OpenMM para prototipado rápido y métodos avanzados en Python.

## 3. Proyecto Integrador: Simulación de Ubiquitina

En este proyecto simularemos la **ubiquitina humana** (PDB: 1UBQ, 76 residuos), una proteína pequeña y bien caracterizada ideal para practicar todos los pasos del flujo de trabajo.

In [ ]:
import requests
from Bio import PDB

# Descargar ubiquitina
def descargar_pdb(pdb_id, directorio='proyecto_ubiquitina'):
    Path(directorio).mkdir(exist_ok=True)
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url)
    if r.status_code == 200:
        p = Path(directorio) / f"{pdb_id}.pdb"
        p.write_text(r.text)
        print(f"✓ Descargado: {p}")
        return p
    return None

def limpiar_pdb_para_dm(entrada, salida):
    """Remover agua y heteroátomos para preparación de DM."""
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure('prot', entrada)
    
    class ProteinSelect(PDB.Select):
        def accept_residue(self, res):
            return res.id[0] == ' '  # solo aminoácidos
    
    io = PDB.PDBIO()
    io.set_structure(struct)
    io.save(str(salida), ProteinSelect())
    print(f"✓ PDB limpio: {salida}")

pdb_raw = descargar_pdb('1UBQ')
if pdb_raw:
    pdb_clean = Path('proyecto_ubiquitina') / '1UBQ_clean.pdb'
    limpiar_pdb_para_dm(pdb_raw, pdb_clean)
    
    # Información de la proteína
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure('ubq', pdb_clean)
    residuos = [r for r in struct.get_residues() if r.id[0] == ' ']
    atomos = list(struct.get_atoms())
    print(f"\nUbiquitina (1UBQ):")
    print(f"  Residuos: {len(residuos)}")
    print(f"  Átomos:   {len(atomos)}")

## 4. Simulación Completa con OpenMM

In [ ]:
# Simulación completa de ubiquitina con OpenMM
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
    from sys import stdout

    print("="*60)
    print("SIMULACIÓN DE UBIQUITINA CON OPENMM")
    print("="*60)

    # 1. Cargar y preparar
    print("\n[1/5] Preparando sistema...")
    pdb = app.PDBFile('proyecto_ubiquitina/1UBQ_clean.pdb')
    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = app.Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(forcefield, pH=7.0)
    modeller.addSolvent(forcefield, model='tip3p',
                        padding=1.0*unit.nanometer,
                        ionicStrength=0.15*unit.molar)
    print(f"  Átomos totales: {modeller.topology.getNumAtoms():,}")

    # 2. Crear sistema
    print("[2/5] Creando sistema físico...")
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometer,
        constraints=app.HBonds
    )

    # 3. Minimización
    print("[3/5] Minimizando energía...")
    integrator = mm.VerletIntegrator(0.001*unit.picoseconds)
    sim = app.Simulation(modeller.topology, system, integrator)
    sim.context.setPositions(modeller.positions)
    E_antes = sim.context.getState(getEnergy=True).getPotentialEnergy()
    sim.minimizeEnergy(maxIterations=500)
    E_despues = sim.context.getState(getEnergy=True).getPotentialEnergy()
    print(f"  E antes:  {E_antes}")
    print(f"  E después: {E_despues}")
    pos_min = sim.context.getState(getPositions=True).getPositions()

    # 4. NVT (10 ps)
    print("[4/5] Equilibración NVT (10 ps)...")
    integrador_nvt = mm.LangevinMiddleIntegrator(
        300*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds
    )
    sim_nvt = app.Simulation(modeller.topology, system, integrador_nvt)
    sim_nvt.context.setPositions(pos_min)
    sim_nvt.context.setVelocitiesToTemperature(300*unit.kelvin)
    sim_nvt.reporters.append(app.StateDataReporter(
        'proyecto_ubiquitina/nvt_log.txt', 1000,
        step=True, temperature=True, potentialEnergy=True
    ))
    sim_nvt.step(5000)  # 10 ps
    pos_nvt = sim_nvt.context.getState(getPositions=True).getPositions()

    # 5. NPT (50 ps producción)
    print("[5/5] Producción NPT (50 ps)...")
    system.addForce(mm.MonteCarloBarostat(1*unit.bar, 300*unit.kelvin))
    integrador_npt = mm.LangevinMiddleIntegrator(
        300*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds
    )
    sim_npt = app.Simulation(modeller.topology, system, integrador_npt)
    sim_npt.context.setPositions(pos_nvt)
    sim_npt.context.setVelocitiesToTemperature(300*unit.kelvin)
    sim_npt.reporters.append(app.StateDataReporter(
        'proyecto_ubiquitina/npt_log.txt', 1000,
        step=True, time=True, temperature=True,
        potentialEnergy=True, density=True
    ))
    sim_npt.reporters.append(app.PDBReporter(
        'proyecto_ubiquitina/traj.pdb', 5000
    ))
    sim_npt.step(25000)  # 50 ps

    print("\n✓ Simulación completada!")
    print("Archivos generados en proyecto_ubiquitina/")

except ImportError:
    print("OpenMM no disponible. Este bloque requiere OpenMM instalado.")
except Exception as e:
    print(f"Error: {e}")

## 5. Análisis Post-Simulación

In [ ]:
# Leer log de OpenMM y analizar
def leer_log_openmm(archivo):
    """
    Lee el archivo de log de StateDataReporter de OpenMM.
    
    Returns:
        DataFrame con columnas del log
    """
    try:
        df = pd.read_csv(archivo, comment=None)
        df.columns = [c.strip('#').strip() for c in df.columns]
        return df
    except Exception:
        return None

# Intentar leer el log real o crear datos simulados
log_file = 'proyecto_ubiquitina/npt_log.txt'
if Path(log_file).exists():
    df = leer_log_openmm(log_file)
    if df is not None:
        print("Log cargado correctamente")
        print(df.head())
else:
    # Datos simulados para demostración
    np.random.seed(42)
    n = 25
    t_ps = np.arange(0, 50, 2.0)
    df = pd.DataFrame({
        'Time (ps)':           t_ps,
        'Temperature (K)':     300 + np.random.normal(0, 5, n),
        'Potential Energy (kJ/mole)': -280000 + np.random.normal(0, 200, n),
        'Density (g/mL)':      0.997 + np.random.normal(0, 0.003, n),
    })
    print("Usando datos simulados para demostración")

# Graficar propiedades
fig, axes = plt.subplots(3, 1, figsize=(10, 10))

t_col = 'Time (ps)'
for col, ax, color, label in [
    ('Temperature (K)',          axes[0], 'blue',  'Temperatura (K)'),
    ('Potential Energy (kJ/mole)', axes[1], 'green', 'Energía Potencial (kJ/mol)'),
    ('Density (g/mL)',            axes[2], 'purple','Densidad (g/mL)'),
]:
    if col in df.columns:
        ax.plot(df[t_col], df[col], f'{color[0]}-o', linewidth=1.5, markersize=4)
        ax.set_ylabel(label, fontsize=12)
        ax.grid(True, alpha=0.3)

axes[2].set_xlabel('Tiempo (ps)', fontsize=12)
plt.suptitle('Monitoreo de Simulación NPT – Ubiquitina', fontsize=14)
plt.tight_layout()
plt.savefig('proyecto_ubiquitina/monitoreo_npt.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Guía de Instalación de GROMACS

In [ ]:
guia_instalacion = """
GUÍA DE INSTALACIÓN DE GROMACS
================================

Opción 1: Conda (más fácil)
----------------------------
conda install -c conda-forge gromacs

# Verificar instalación
gmx --version


Opción 2: Docker (sin instalación)
------------------------------------
docker pull gromacs/gromacs
docker run -it --rm -v $(pwd):/data gromacs/gromacs gmx --version


Opción 3: Google Colab
-----------------------
!apt-get install -y gromacs


Opción 4: Compilar desde fuente (con GPU CUDA)
-----------------------------------------------
cmake .. -DGMX_BUILD_OWN_FFTW=ON -DREGRESSIONTEST_DOWNLOAD=ON -DGMX_GPU=CUDA
make -j 8
make check
sudo make install


GUÍA DE INSTALACIÓN DE OPENMM
================================

Opción 1: Conda (recomendado)
-------------------------------
conda install -c conda-forge openmm

# Verificar
python -m openmm.testInstallation


Opción 2: pip
-------------
pip install openmm


ENTORNO CONDA COMPLETO PARA MÓDULO 5
======================================
conda create -n dm_env python=3.11
conda activate dm_env
conda install -c conda-forge gromacs openmm mdanalysis biopython numpy matplotlib pandas scipy seaborn
pip install py3Dmol
"""
print(guia_instalacion)

Path('proyecto_ubiquitina').mkdir(exist_ok=True)
with open('proyecto_ubiquitina/INSTALACION.md', 'w') as f:
    f.write("# Guía de Instalación\n\n" + guia_instalacion)
print("Guía guardada en: proyecto_ubiquitina/INSTALACION.md")

## 7. Proyecto Final del Módulo

### Instrucciones

Realiza una simulación de DM completa de la proteína de tu elección y presenta un informe con:

1. **Introducción** (1 párrafo): importancia biológica de la proteína elegida
2. **Metodología**:
   - Software utilizado (GROMACS y/o OpenMM)
   - Campo de fuerza y modelo de agua
   - Dimensiones de la caja y número de átomos
   - Protocolo de equilibración y tiempo de producción
3. **Resultados y Análisis**:
   - Gráficas de RMSD, RMSF y Rg
   - Número de puentes de hidrógeno
   - Temperatura y densidad durante la simulación
   - Al menos una observación biológicamente relevante
4. **Conclusiones** (1 párrafo)

### Proteínas sugeridas

| Proteína | PDB | Residuos | Dificultad |
|----------|-----|----------|------------|
| Villin headpiece | 1VII | 35 | Fácil |
| Trp-cage | 1L2Y | 20 | Fácil |
| Ubiquitina | 1UBQ | 76 | Media |
| Lisozima | 1AKI | 129 | Media |
| DHFR + metotrexato | 1DRF | 186 | Avanzado |

In [ ]:
# Plantilla de informe en Markdown
plantilla_informe = """
# Informe: Simulación de Dinámica Molecular de [NOMBRE_PROTEÍNA]

**Autor:** [Tu nombre]  
**Fecha:** [Fecha]  
**Código PDB:** [XXXX]  

---

## 1. Introducción
[Describe brevemente la proteína y su importancia biológica]

## 2. Metodología

### Software
- Software de DM: [GROMACS X.X / OpenMM X.X]
- Campo de fuerza: [AMBER14SB / CHARMM36m]
- Modelo de agua: [TIP3P / SPC/E]

### Sistema
| Parámetro | Valor |
|-----------|-------|
| Residuos de aminoácidos | |
| Átomos totales | |
| Tipo de caja | |
| Dimensiones de la caja | |
| Moléculas de agua | |
| Iones (Na⁺/Cl⁻) | |

### Protocolo de Simulación
| Etapa | Tiempo | Ensamble | Termostato | Barostato |
|-------|--------|----------|------------|-----------|
| Minimización | - | - | - | - |
| NVT | ps | NVT | V-rescale | - |
| NPT equil. | ps | NPT | V-rescale | Parrinello-Rahman |
| Producción | ns | NPT | V-rescale | Parrinello-Rahman |

## 3. Resultados

### 3.1 RMSD
[Insertar gráfica y análisis]

### 3.2 RMSF
[Insertar gráfica y análisis]

### 3.3 Radio de Giro
[Insertar gráfica y análisis]

### 3.4 Propiedades Termodinámicas
[Temperatura, densidad, energía potencial]

## 4. Conclusiones
[Describe los principales hallazgos de la simulación]

## 5. Referencias
1. [Referencia del campo de fuerza]
2. [Referencia del software]
3. [Referencia de la proteína]
"""

with open('proyecto_ubiquitina/plantilla_informe.md', 'w') as f:
    f.write(plantilla_informe)
print("Plantilla guardada en: proyecto_ubiquitina/plantilla_informe.md")

## 8. Resumen del Módulo 5

En este módulo hemos cubierto el flujo completo de dinámica molecular:

| Actividad | Tema | Herramientas |
|-----------|------|--------------|
| 5.1 | Fundamentos de DM | Python, NumPy, Matplotlib |
| 5.2 | Integradores (Verlet, LF, VV) | Python, NumPy |
| 5.3 | PBC y Ensambles | Python, NumPy |
| 5.4 | Termostatos y Barostatos | Python, GROMACS mdp |
| 5.5 | Preparación de Sistemas | GROMACS, OpenMM, BioPython |
| 5.6 | Simulación de Proteínas | GROMACS, OpenMM |
| 5.7 | Análisis de Trayectorias | MDAnalysis, GROMACS tools |
| 5.8 | Proyecto Integrador | GROMACS + OpenMM |

### Próximos pasos

Después de dominar la DM clásica, puedes explorar métodos avanzados:
- **Enhanced sampling:** metadinámica, replica exchange (REMD), steered MD
- **Energía libre:** FEP, TI, umbrella sampling
- **Coarse-graining:** MARTINI, modelos de bead
- **Machine learning force fields:** ANI, MACE, NequIP

## 9. Recursos Adicionales

### Software
- [GROMACS](https://www.gromacs.org/) — Motor de DM de alto rendimiento
- [OpenMM](https://openmm.org/) — DM en Python con GPU
- [NAMD](https://www.ks.uiuc.edu/Research/namd/) — DM para sistemas grandes
- [AMBER](https://ambermd.org/) — Suite completa de DM biomolecular
- [VMD](https://www.ks.uiuc.edu/Research/vmd/) — Visualización de trayectorias

### Tutoriales
- [GROMACS Tutorials (Justin Lemkul)](http://www.mdtutorials.com/gmx/)
- [OpenMM User Guide](https://openmm.org/documentation/latest/userguide/)
- [MDAnalysis Tutorials](https://www.mdanalysis.org/MDAnalysisTutorial/)
- [BioExcel Training](https://bioexcel.eu/training/)

### Bases de datos y recursos
- [RCSB PDB](https://www.rcsb.org/) — Estructuras de proteínas
- [CHARMM-GUI](https://www.charmm-gui.org/) — Preparación de sistemas
- [SWISS-MODEL](https://swissmodel.expasy.org/) — Modelado por homología

### Libros de referencia
- Frenkel & Smit, *Understanding Molecular Simulation* (2002)
- Allen & Tildesley, *Computer Simulation of Liquids* (2017)
- Tuckerman, *Statistical Mechanics: Theory and Molecular Simulation* (2010)
- Leach, *Molecular Modelling: Principles and Applications* (2001)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Ejecutar de forma autónoma un protocolo completo de DM con GROMACS y/o OpenMM
- ✅ Comparar el desempeño y las capacidades de ambos softwares
- ✅ Analizar e interpretar los resultados de la simulación con herramientas integradas
- ✅ Evaluar la calidad de la simulación y detectar posibles artefactos
- ✅ Generar un informe científico con los resultados del análisis

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.8: Práctica con GROMACS y OpenMM**

¡Has finalizado el **Módulo 5: Dinámica Molecular** completo! 🚀

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.7-Análisis_de_Trayectorias-blue.svg)](07_analisis_trayectorias.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>